# 2. Train: the released `noise` model

This notebook trains the released `noise` model (`swin_t_cnn_noise_10-20`) the way it was originally
trained:
- **Model:** a Swin-T backbone (starting from ImageNet weights) with CNN experts, on the
  `mnr_17leaf` taxonomy
- **Data:** up to 12,000 images per class from the class folders in `/data/Hier-Zooplankton` and
  `IssacData/`, plus all of `Quetico_training_data/`
- **Augmentation:** synthetic background noise (specks and fibres) on 80% of training images, plus
  random flips
- **Schedule:** 40 epochs, keeping the checkpoint with the best validation accuracy

It takes about 20 minutes on one GPU. `inference.ipynb` then tests the result on the Quetico
messy-background test set, next to the released weights.

**The result will be close to the released model but not identical.** Training on a GPU isn't fully
deterministic: re-running the original training script with the same seed doesn't reproduce its own
numbers exactly either. The last section compares this run with the released one.

To check that everything works before committing to the full run, set `QUICK = True`. That trains on
300 images per class for 3 epochs and takes a few minutes.

Requires `pip install hierroute jupyter`. The example data lives at `/data/Hier-Zooplankton` on the lab server; change `DATA_DIR` below if your copy is elsewhere. Everything this notebook writes goes under `WORK_DIR`.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/data/Hier-Zooplankton")        # example dataset (internal)
WORK_DIR = Path.home() / "hierroute_example"      # everything the example notebooks write goes here
WORK_DIR.mkdir(parents=True, exist_ok=True)

QUICK = False                           # True: 300 images per class, 3 epochs (smoke test)
RUN_NAME = "swin_t_cnn_noise_example"   # run folder under WORK_DIR / "training_result"

## The taxonomy and how folders map onto it

A **hierarchy** defines the classes and how they nest. The released models use the bundled
`mnr_17leaf` hierarchy:
- every leaf class (one with no children) is read from a folder of the same name
- `folder_map` pools several folders into one class (e.g. `Bosminidae`, `Eubosmina` → `Bosmina`)

To use your own taxonomy, write a YAML file in the same format (see the README) and pass its path
instead of `"mnr_17leaf"`.

In [ ]:
from hierroute.config import load_hierarchy

spec = load_hierarchy("mnr_17leaf")

def show(node, depth=0):
    print("    " * depth + node)
    for child in spec.tree[node]:
        show(child, depth + 1)

show("root")
print("\nfolder_map:")
for cls, folders in spec.folder_map.items():
    print(f"  {cls:18s} <- {folders}")

## The training data

Training reads one folder per class from several places under `DATA_DIR`:

| Source | Setting | Use |
|---|---|---|
| class folders at the top of `DATA_DIR` | `data_dir` | capped at `max_class_size` per class, together with `IssacData` |
| `IssacData/` | `subdirectories` | same layout, pooled in |
| `Quetico_training_data/` | `uncapped_subdirectories` | Quetico debris and bubbles on messy backgrounds, always included in full |
| `Messy_background/moved_images/log.md` | `exclude_log` | lists images moved out to build a messy-background test set; they are skipped |

The table below counts the files on disk, before the cap and exclusions. The numbers actually used are
printed when training starts.

In [ ]:
import pandas as pd

def count(base, cls):
    return sum(len(list((base / f).glob("*.tif"))) for f in spec.folder_map.get(cls, [cls])
               if (base / f).is_dir())

sources = {"top level": DATA_DIR, "IssacData": DATA_DIR / "IssacData",
           "Quetico_training_data": DATA_DIR / "Quetico_training_data"}
counts = pd.DataFrame({name: {cls: count(base, cls) for cls in spec.leaves} for name, base in sources.items()})
counts["total"] = counts.sum(axis=1)
counts.sort_values("total")

## Configure the run

These are the released model's settings, the same as `configs/mnr_swin_t_cnn_noise.yaml` in the
repository. `load_train_config` starts from the defaults (`TRAIN_DEFAULTS` in `hierroute/config.py`)
and applies the overrides. All settings are documented in the README.

The data is split 70 / 10 / 20 into train / validation / test, stratified by class with seed 666. With
the same data and settings you get the same split as the released model, so the test scores at the end
are directly comparable.

In [ ]:
import yaml
from hierroute.config import load_train_config

config = load_train_config(overrides={
    "hierarchy": "mnr_17leaf",
    "model": {"backbone": "swin_t", "expert": "cnn", "loss": "bce"},
    "data": {
        "data_dir": str(DATA_DIR),
        "subdirectories": ["IssacData"],
        "uncapped_subdirectories": ["Quetico_training_data"],
        "exclude_log": "Messy_background/moved_images/log.md",
        "resolution": 64,
        "max_class_size": 12000,
        "split": [0.7, 0.1, 0.2],
        "balanced_sampling": True,        # inverse-frequency sampling of the training split
    },
    "augment": {"background_noise": True, "noise_p": 0.8, "hflip": True, "vflip": True},
    "train": {"lr": 3e-4, "max_epochs": 40, "scheduler": "cosine", "patience": 10, "delta": 0.001,
              "gradient_clip": 1.0, "batch_size": 64, "num_workers": 8, "seed": 666},
    "out_dir": str(WORK_DIR / "training_result"),
    "run_name": RUN_NAME,
})
if QUICK:
    config["data"]["max_class_size"] = 300
    config["train"]["max_epochs"] = 3

print(yaml.safe_dump(config, sort_keys=False))

## Train

`train` builds the dataset, splits it, trains, then scores the held-out test split with the best
checkpoint. It returns the run folder, which contains:
- `best_model.pt`, the checkpoint
- `model_config.json`, so it loads like a released model
- `train_config.yaml`, `training_info.json`, `eval_results.json`, `predictions.npz`

An existing run folder is never overwritten; a `_1`, `_2`, … suffix is added instead. Delete an old
run (or change `RUN_NAME`) if you want `inference.ipynb` to pick up the new one.

In [ ]:
from hierroute.train import train

run_dir = train(config)
run_dir

## Results

In [ ]:
from hierroute import Visualize

vis = Visualize(run_dir)
vis.plot_train()          # loss / accuracy / F1 per epoch

Test-split scores of this run next to the released model's, on the same 14,345 test images. Expect
differences in the second or third decimal place.

In [ ]:
import json

# Released swin_t_cnn_noise_10-20: test-split scores (eval_results.json) and best epoch (training_info.json)
RELEASED = {1: {"accuracy": 0.9928, "f1": 0.9883},
            2: {"accuracy": 0.9872, "f1": 0.7412},
            3: {"accuracy": 0.9473, "f1": 0.9077}}
RELEASED_BEST = {"best_epoch": 38, "best_val_acc": 0.9543}

levels = json.loads((run_dir / "eval_results.json").read_text())["level_results"]
table = pd.DataFrame({
    f"level {d}": {"accuracy (this run)": r["overall"]["accuracy"],
                   "accuracy (released)": RELEASED[int(d)]["accuracy"],
                   "F1 (this run)": r["overall"]["f1"],
                   "F1 (released)": RELEASED[int(d)]["f1"],
                   "test images": r["overall"]["n_samples"]}
    for d, r in levels.items()}).T
best = json.loads((run_dir / "training_info.json").read_text())["results"]
print(f"best epoch {best['best_epoch']} (released {RELEASED_BEST['best_epoch']}), "
      f"best validation accuracy {best['best_val_acc']:.4f} (released {RELEASED_BEST['best_val_acc']})")
table.round(4)

## Next

`inference.ipynb` evaluates this run and the released weights on the Quetico messy-background test set.
Any trained run is used exactly like a released model, by passing its folder as `weights`:

```bash
hierroute evaluate /data/Hier-Zooplankton/quetico/Messy --labelled \
    --weights ~/hierroute_example/training_result/swin_t_cnn_noise_example --out results/
```

The same training from the terminal, using the config file in the repository:

```bash
hierroute train --config configs/mnr_swin_t_cnn_noise.yaml --data-dir /data/Hier-Zooplankton \
    --out-dir ~/hierroute_example/training_result --run-name swin_t_cnn_noise_example
```